# 🦜 Personal Resource Assistant — RAG with LangChain, Cohere & ChromaDB — Explained

**What this notebook does:** builds a real, working RAG (Retrieval-Augmented Generation) system end to end — extracts text from PDF research papers, splits it into overlapping chunks, embeds and stores them in a persistent ChromaDB vector database, then wires retrieval together with a Cohere chat model using LCEL to answer questions grounded in those papers. Also demonstrates the three ways to run any LCEL chain: `.invoke()`, `.batch()`, and `.stream()`. Each code cell below has a short explanation directly above it, in addition to the notebook author's own original notes.

## Installing necessary libraries

Cohere provides free trial keys to use their LLMs. So generate one trial key from dashboard.cohere.com

### 📦 Cell 2 — Install the libraries

Installs everything this RAG assistant needs: LangChain's Cohere integration, core LangChain, `pdfminer.six` for pulling text out of PDFs, `chromadb` for the vector database, plus community integrations and a text-splitter package. (The markdown note right after this cell already explains what each package is for.)

In [ ]:
!pip install langchain-cohere langchain pdfminer.six chromadb langchain-community langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 k

langchain-cohere: Enables integration of Cohere's language models with LangChain for advanced text generation and processing workflows.

langchain: Provides a modular framework for building language model-powered applications, such as chatbots, question-answering systems, and conversational agents.

pdfminer.six: Facilitates text extraction from PDF files, making it useful for document analysis and preprocessing tasks.

chromadb: A vector database library designed for efficient storage and retrieval of embeddings, ideal for tasks like semantic search and recommendation systems.

## Importing libraries

### 🧰 Cell 5 — Import everything, and set the Cohere API key

Reads the Cohere API key from Colab's secret storage and sets it as an environment variable — `ChatCohere` and `CohereEmbeddings` pick this up automatically, so it never has to be passed in by hand later. The rest of the imports bring in every building block this notebook uses: prompt templates, the Cohere chat model, an output parser, the PDF text extractor, Chroma (the vector store), the Cohere embedding model, a text splitter, LangChain's `Document` wrapper, and `RunnableParallel` / `RunnablePassthrough` — two LCEL helpers used later to fetch context and the question side by side.

In [ ]:
import os
from google.colab import userdata
os.environ["COHERE_API_KEY"] = userdata.get('COHERE_KEY')
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_cohere import ChatCohere
from langchain_core.output_parsers import StrOutputParser
from pdfminer.high_level import extract_text as extract_text_pdf_miner
from langchain_community.vectorstores import Chroma
from langchain_cohere import CohereEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.runnables import RunnableParallel,RunnablePassthrough

## VectorDB setup

### 🗄️ Cell 7 — Set up where the vector database lives, and which embedding model to use

`persist_directory` is the folder on disk where Chroma saves its data — the same idea as `PersistentClient` from the ChromaDB tutorial notebook, so the database survives a restart. `CohereEmbeddings` wraps Cohere's `embed-english-v3.0` embedding model so LangChain can use it — every PDF chunk, and every question later, gets turned into a vector by this exact same model (the “same model on both sides” rule from the vector database notes).

In [ ]:
# Define the directory where the Chroma database will persist data
persist_directory = "/content/chroma_db"

# Initialize Cohere embeddings with the specified model
# "embed-english-v3.0" is a pre-trained English language embedding model by Cohere
# The user_agent parameter specifies the tool or library using the Cohere API, in this case, LangChain
embedding = CohereEmbeddings(
    model="embed-english-v3.0",
    user_agent="langchain"
)


We are processing 2 research papers on transformers and yolo. You can use your own PDFs.

### 📄 Cell 9 — Turn each PDF into chunks, and store them in the vector database

For each PDF: `extract_text_pdf_miner` pulls the raw text out; the text is cleaned by joining everything onto one line, removing stray newlines left over from the PDF extraction; `RecursiveCharacterTextSplitter` then cuts that long text into chunks of 2048 characters each, with 512 characters of overlap between consecutive chunks — the same chunking-with-overlap idea from the RAG notes, so an idea sitting right at a chunk boundary doesn't get cut in half. Each chunk is wrapped in a `Document`, tagged in its metadata with which PDF it came from. Finally, `Chroma.from_documents(...)` embeds every chunk with the Cohere embedding model and saves it all into the persistent vector database.

**Example:** a paper on Transformers gets sliced into many overlapping 2048-character chunks, each becoming its own entry in the vector database, tagged `{'source': '/content/1706.03762v7.pdf'}` so an answer can always be traced back to which paper it came from.

In [ ]:
# Loop through a list of PDF files to process
for pdf_name in ["/content/1706.03762v7.pdf", "/content/1506.02640v5.pdf"]:
    # Open each PDF file in binary mode
    with open(pdf_name, 'rb') as f:
        # Extract text from the PDF using the extract_text_pdf_miner function
        text = extract_text_pdf_miner(f)

        # Clean the extracted text by removing newline characters and joining into a single string
        cleaned_text = " ".join(text.split("\n"))

        # Initialize a list to store document chunks
        docs = []

        # Create a text splitter to divide the text into manageable chunks
        # Each chunk has a maximum size of 2048 characters with a 512-character overlap
        splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=512)

        # Split the cleaned text into chunks and wrap each chunk in a Document object
        for chunk in splitter.split_text(cleaned_text):
            docs.append(Document(page_content=chunk, metadata={"source": pdf_name}))

    # Create a Chroma collection from the processed documents
    # Use the specified persist directory and embedding model for storage and retrieval
    vector_collection_fixed_size = Chroma.from_documents(
        documents=docs,
        persist_directory=persist_directory,
        embedding=embedding
    )

### 🔌 Cell 10 — Reconnect to the vector database

Opens a handle to the *same* persistent Chroma database that was just filled with chunks in Cell 9 — using the same `persist_directory` and the same embedding model, so future searches compare apples to apples.

In [ ]:
# Initialize a Chroma vector database
# The persist_directory specifies the location where the database is stored
# The embedding_function parameter provides the embedding model used for vector representation
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)

/tmp/ipykernel_2485/1816146847.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)


### 🔍 Cell 11 — Try a similarity search directly

A quick sanity check before building the full pipeline: search the vector database directly for the single closest chunk (`k=1`) to the question “What is YOLO?”, along with a relevance score showing how close the match actually is. This is dense vector retrieval in its rawest form — no prompt, no LLM involved yet, just “find me the closest chunk.”

In [ ]:
# Perform a similarity search on the vector database
# The query "What is YOLO?" is used to find the most relevant documents
# k=1 specifies that the top 1 most similar document should be retrieved
# The method also returns relevance scores indicating how closely each document matches the query
vectordb.similarity_search_with_relevance_scores("What is YOLO?", k=1)

[(Document(metadata={'source': '/content/1506.02640v5.pdf'}, page_content='Detection In The Wild  Academic datasets for object detection draw the training and testing data from the same distribution. In real-world applications it is hard to predict all possible use cases and  YOLO is a fast, accurate object detector, making it ideal for computer vision applications. We connect YOLO to a webcam and verify that it maintains real-time performance,  \x0cVOC 2007 AP 59.2 54.2 43.2 36.5 -  Picasso AP Best F1 0.590 53.3 0.226 10.4 0.458 37.8 0.271 17.8 0.051 1.9  People-Art AP 45 26 32  YOLO R-CNN DPM Poselets [2] D&T [4]  (a) Picasso Dataset precision-recall curves.  (b) Quantitative results on the VOC 2007, Picasso, and People-Art Datasets. The Picasso Dataset evaluates on both AP and best F1 score.  Figure 5: Generalization results on Picasso and People-Art datasets.  Figure 6: Qualitative Results. YOLO running on sample artwork and natural images from the internet. It is mostly accurate a

## RAG pipeline

### 🔗 Cell 13 — Build the full RAG pipeline with LCEL

This is where retrieval and generation actually get wired together, piece by piece:

- **`llm`** — Cohere's `command-a-plus-05-2026` model, with `temperature=0` so its answers come out consistent and deterministic rather than creative or random.
- **`prompt_str` / `prompt`** — a template with two placeholders, `{context}` and `{question}`, instructing the model to answer strictly using the given context.
- **`retrieval`** — the new piece: `RunnableParallel` runs two things *side by side* rather than one after another — `vectordb.as_retriever()` turns the vector database into something that can be piped directly into a chain (it embeds the incoming question and runs the similarity search automatically), while `RunnablePassthrough()` just lets the original question through untouched. The result is a dictionary, `{"context": <retrieved chunks>, "question": <the original question>}` — exactly the two placeholders `prompt` expects.
- **`chain = retrieval | prompt | llm | output_parser`** — the full LCEL pipeline: fetch context and pass through the question together, fill both into the prompt, send it to the model, then parse the raw response into plain text.

**Example:** ask “What is YOLO?” — `retrieval` fetches the most relevant chunks about YOLO from the vector database *and* keeps the question itself, `prompt` weaves both into one instruction, `llm` reads that instruction and drafts an answer grounded in the retrieved text, and `output_parser` hands back just the plain answer.

In [ ]:
# Initialize an LLM instance using Cohere's "command-r" model
# The temperature parameter controls randomness in the generated responses; 0 ensures deterministic outputs
llm = ChatCohere(model="command-a-plus-05-2026", temperature=0)

# Define a prompt template for generating answers based on a given context and question
prompt_str = """Answer the question below using the context:

Context: {context}

Question: {question}

Answer: """

# Create a ChatPromptTemplate from the string template, enabling dynamic input for context and question
prompt = ChatPromptTemplate.from_template(prompt_str)

# Create a retrieval pipeline to fetch relevant context and pass through the user's question
retrieval = RunnableParallel(
    {
        # Use the vector database as a retriever to fetch relevant context for the question
        "context": vectordb.as_retriever(),

        # Pass through the user's input question without modification
        "question": RunnablePassthrough()
    }
)

# Define an output parser to format the generated response into a string
output_parser = StrOutputParser()

# Create a processing chain that retrieves context, formats the prompt, generates an LLM response, and parses the output
chain = retrieval | prompt | llm | output_parser

### ▶️ Cell 14 — Run the full RAG pipeline

`.invoke("What is YOLO?")` runs the entire chain from Cell 13 end to end — retrieve the relevant chunks, build the prompt, generate an answer, parse it — and `response` holds just the final text.

In [ ]:
# Invoke the chain of components (retrieval, prompt generation, LLM processing, and output parsing)
# The question "What is YOLO?" is passed through the chain to generate the response
response = chain.invoke("What is YOLO?")

# Print the response generated by the chain
print(response)

YOLO is a unified, end‑to‑end object detection model that uses a single convolutional network to look once at an image and directly predict multiple bounding boxes and class probabilities. It is designed to be extremely fast (running at 45–150 fps) and to generalize well to new domains, making it ideal for real‑time applications such as webcam‑based detection.


## Other chain invoking methods!

.invoke(): The goal is to pass in an input and receive the output—neither more nor less.

.batch(): This is faster than using invoke three times when you wish to supply several inputs to get multiple outputs because it handles the parallelization for you.

.stream():  We may begin printing the response before the entire response is complete.

### 📚 Cell 16 — Run the chain on multiple questions at once

`.batch([...])` runs the *same* chain over several questions in one call, and LangChain handles running them efficiently rather than looping through `.invoke()` one at a time. `response_with_batch` comes back as a list, one answer per question, in the same order they were asked.

In [ ]:
response_with_batch = chain.batch(["What is Transformers", "How is Transformer different than YOLO?"])

for response in response_with_batch:
  print(response)
  print("\n")

Transformers are a neural architecture for sequence transduction that replaces recurrence with an attention mechanism. Specifically, the Transformer is the first model that relies entirely on self‑attention (and multi‑head attention) to compute representations of its input and output, without using any sequence‑aligned RNNs or convolutions. It follows an encoder‑decoder structure where both encoder and decoder consist of stacked identical layers, each containing a multi‑head self‑attention sub‑layer followed by a position‑wise fully connected feed‑forward network, residual connections, and layer normalization. This design allows the model to draw global dependencies between any input and output positions with a constant number of operations, enabling significant parallelization and faster training while achieving state‑of‑the‑art performance on tasks such as machine translation.


The provided context discusses the YOLO object detection system, its architecture, and its advantages over

### 🌊 Cell 17 — Stream the answer as it's generated

`.stream(...)` doesn't wait for the whole answer to be ready before returning anything — it yields small pieces (`chunk`) as the model generates them, which is why they're printed with `end=""` here, so the words appear one after another instead of one whole block appearing all at once. This is exactly the “show partial answers as they're generated” capability that comes for free with any LCEL chain, mentioned in the earlier LCEL notes.

In [ ]:
for chunk in chain.stream("What are the 3 vectors in Transformers architecture?"):
  print(chunk, flush=True, end="")

In the Transformer architecture, each attention head projects the input vectors into three distinct vectors that drive the attention computation:

- **Query (Q)** – represents the current position’s query for relevant information.
- **Key (K)** – represents the positions that can be attended to; compatibility is measured between queries and keys.
- **Value (V)** – holds the information that is actually retrieved via the attention weights.

These three vectors (Q, K, V) are obtained by linear projections of the input and are used to compute attention scores (typically via dot‑product) and then to produce a weighted sum of the values, which becomes the output of the attention sub‑layer.